In [1]:
import warnings
import numpy as np
# import lightgbm as lgb
# from fontTools.misc.cython import returns
# from pyarrow.types import is_large_binary
# from sympy.codegen.ast import continue_
# from xgboost import XGBRegressor
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.svm import SVR
# from sklearn.neural_network import MLPRegressor
# from sklearn.tree import DecisionTreeRegressor
# from statsmodels.tools.eval_measures import rmse, hqic_sigma
import pandas as pd
import re
# from optimize_params import *
import matplotlib.pyplot as plt
from datetime import datetime

# np.random.seed(42)

warnings.filterwarnings("ignore", category=RuntimeWarning)

def mape(y_true, y_pred):
    """
    Calculate Mean Absolute Percentage Error (MAPE)

    Parameters:
        y_true (array-like): Actual values
        y_pred (array-like): Predicted values

    Returns:
        float: MAPE in percentage (%)
    """
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    # Avoid division by zero
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

In [2]:
train = pd.read_parquet("../data/money_calc/train.parquet")

train = train.reset_index(drop=True)

In [3]:
import pandas as pd
import numpy as np

def smallest_int_dtype(min_val: int, max_val: int, signed: bool = True) -> str:
    if signed:
        if np.iinfo(np.int8).min <= min_val <= max_val <= np.iinfo(np.int8).max:
            return "int8"
        if np.iinfo(np.int16).min <= min_val <= max_val <= np.iinfo(np.int16).max:
            return "int16"
        if np.iinfo(np.int32).min <= min_val <= max_val <= np.iinfo(np.int32).max:
            return "int32"
        return "int64"
    else:
        if 0 <= min_val <= max_val <= np.iinfo(np.uint8).max:
            return "uint8"
        if 0 <= min_val <= max_val <= np.iinfo(np.uint16).max:
            return "uint16"
        if 0 <= min_val <= max_val <= np.iinfo(np.uint32).max:
            return "uint32"
        return "uint64"


def optimize_df_for_memory(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """
    Converts columns to smaller dtypes.
    For low-decimal float columns, stores scaled integers if that beats float32.
    Returns:
        optimized_df
        metadata dict with scaling info
    """
    df = df.copy()
    meta = {}

    for col in df.columns:
        s = df[col]

        # bool-like columns
        unique_non_null = set(s.dropna().unique())
        if unique_non_null.issubset({0, 1, True, False}):
            if col == "Група":
                df[col] = s.astype("bool")
                meta[col] = {"stored_as": "bool", "scale": 1}
                continue

        # integer columns
        if pd.api.types.is_integer_dtype(s):
            mn, mx = int(s.min()), int(s.max())
            dtype = smallest_int_dtype(mn, mx, signed=(mn < 0))
            df[col] = s.astype(dtype)
            meta[col] = {"stored_as": dtype, "scale": 1}
            continue

        # float columns
        if pd.api.types.is_float_dtype(s):
            # estimate visible decimal precision
            non_null = s.dropna()
            if len(non_null) == 0:
                df[col] = s.astype("float32")
                meta[col] = {"stored_as": "float32", "scale": 1}
                continue

            decimals = non_null.astype(str).apply(
                lambda x: len(x.split(".")[1].rstrip("0")) if "." in x else 0
            ).max()

            # try scaled integer
            if decimals <= 3:
                scale = 10 ** decimals
                scaled = np.round(s * scale)

                mn = int(np.nanmin(scaled))
                mx = int(np.nanmax(scaled))
                int_dtype = smallest_int_dtype(mn, mx, signed=(mn < 0))

                int_bytes = np.dtype(int_dtype).itemsize
                float32_bytes = np.dtype("float32").itemsize

                if int_bytes < float32_bytes:
                    df[col] = scaled.astype(int_dtype)
                    meta[col] = {"stored_as": int_dtype, "scale": scale}
                else:
                    df[col] = s.astype("float32")
                    meta[col] = {"stored_as": "float32", "scale": 1}
            else:
                df[col] = s.astype("float32")
                meta[col] = {"stored_as": "float32", "scale": 1}

    return df, meta

train, train_meta = optimize_df_for_memory(train)

In [4]:
y_col = 'Money_spent'

In [5]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

2.6.0+cu124
12.4
True


In [6]:
import pandas as pd
import numpy as np
from autogluon.tabular import TabularPredictor
from autogluon.core.metrics import make_scorer
from sklearn.metrics import mean_squared_error

def smape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    return np.mean(np.abs(y_true - y_pred) / np.maximum(denom, 1e-8))


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def mape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8)))


smape_scorer = make_scorer(
    name="rmse",
    score_func=rmse,
    optimum=0,
    greater_is_better=False
)

# Optional but strongly recommended:
# keep a smaller subset while debugging
# train = train.sample(1_000_000, random_state=42).reset_index(drop=True)

predictor = TabularPredictor(
    label=y_col,
    problem_type="regression",
    eval_metric=smape_scorer,
    path="models/autogluon_safe"
)

predictor = TabularPredictor(
    label=y_col,
    problem_type="regression",
    eval_metric=smape_scorer,
    path="models/autogluon_gpu",  # new path to avoid any cached state
    verbosity=3,                  # will log "Fitting X with num_gpus: 1"
)

predictor.fit(
    train_data=train,
    presets="best_quality",
    num_gpus=1,
    dynamic_stacking=False,
    num_bag_folds=0,
    num_stack_levels=0,
    # time_limit=5 * 60,
    # hyperparameters=hp,
    ag_args_fit={
        "ag.max_memory_usage_ratio": 2.0,  # allow up to 2x estimated memory
    },
)

Verbosity: 3 (Detailed Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.9.25
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          12
GPU Count:          1
Memory Avail:       18.80 GB / 31.11 GB (60.4%)
Disk Space Avail:   44.51 GB / 475.82 GB (9.4%)
Presets specified: ['best_quality']
============ fit kwarg info ============
User Specified kwargs:
{'ag_args_fit': {'ag.max_memory_usage_ratio': 2.0},
 'auto_stack': True,
 'num_bag_folds': 0,
 'num_bag_sets': 1,
 'num_stack_levels': 0}
Full kwargs:
{'_experimental_dynamic_hyperparameters': False,
 '_feature_generator_kwargs': None,
 '_save_bag_folds': None,
 'ag_args': None,
 'ag_args_ensemble': None,
 'ag_args_fit': {'ag.max_memory_usage_ratio': 2.0},
 'auto_stack': True,
 'calibrate': 'auto',
 'delay_bag_sets': False,
 'ds_args': {'clean_up_fits': True,
             'detection_time_frac': 0.25,
             'enable_callbacks'

[50]	valid_set's l2: 3179.48	valid_set's rmse: -56.3869
[100]	valid_set's l2: 2467.61	valid_set's rmse: -49.675
[150]	valid_set's l2: 2173.23	valid_set's rmse: -46.618
[200]	valid_set's l2: 2013.43	valid_set's rmse: -44.8713
[250]	valid_set's l2: 1891.26	valid_set's rmse: -43.4887
[300]	valid_set's l2: 1791.79	valid_set's rmse: -42.3295
[350]	valid_set's l2: 1721.62	valid_set's rmse: -41.4924
[400]	valid_set's l2: 1644.08	valid_set's rmse: -40.5472
[450]	valid_set's l2: 1589.31	valid_set's rmse: -39.8661
[500]	valid_set's l2: 1559.91	valid_set's rmse: -39.4957
[550]	valid_set's l2: 1528.15	valid_set's rmse: -39.0916
[600]	valid_set's l2: 1480.23	valid_set's rmse: -38.4737
[650]	valid_set's l2: 1448.7	valid_set's rmse: -38.0619
[700]	valid_set's l2: 1426.29	valid_set's rmse: -37.7663
[750]	valid_set's l2: 1385.52	valid_set's rmse: -37.2225
[800]	valid_set's l2: 1349.09	valid_set's rmse: -36.7299
[850]	valid_set's l2: 1327.77	valid_set's rmse: -36.4385
[900]	valid_set's l2: 1300.65	valid

Saving C:\Users\Lev\Documents\GitHub\Diploma\models\autogluon_gpu\models\LightGBMXT\model.pkl
Saving C:\Users\Lev\Documents\GitHub\Diploma\models\autogluon_gpu\utils\attr\LightGBMXT\y_pred_proba_val.pkl
	-26.7274	 = Validation score   (-rmse)
	442.48s	 = Training   runtime
	1.87s	 = Validation runtime
	25968.4	 = Inference  throughput (rows/s | 48644 batch size)
Saving C:\Users\Lev\Documents\GitHub\Diploma\models\autogluon_gpu\models\trainer.pkl
Fitting model: LightGBM ... Training model for up to 3113.10s of the 3113.07s of remaining time.
	Fitting LightGBM with 'num_gpus': 1, 'num_cpus': 6
	Fitting with cpus=6, gpus=1, mem=13.7/16.7 GB
	Training LightGBM with GPU, note that this may negatively impact model quality compared to CPU training.
	Fitting 10000 rounds... Hyperparameters: {'learning_rate': 0.05, 'device': 'gpu'}


[50]	valid_set's l2: 3081.92	valid_set's rmse: -55.515
[100]	valid_set's l2: 2276.24	valid_set's rmse: -47.7099
[150]	valid_set's l2: 1949.73	valid_set's rmse: -44.1558
[200]	valid_set's l2: 1749.96	valid_set's rmse: -41.8325
[250]	valid_set's l2: 1614.63	valid_set's rmse: -40.1824
[300]	valid_set's l2: 1506.35	valid_set's rmse: -38.8118
[350]	valid_set's l2: 1425.64	valid_set's rmse: -37.7577
[400]	valid_set's l2: 1375.84	valid_set's rmse: -37.0923
[450]	valid_set's l2: 1324.84	valid_set's rmse: -36.3983
[500]	valid_set's l2: 1287.02	valid_set's rmse: -35.8751
[550]	valid_set's l2: 1246.08	valid_set's rmse: -35.2998
[600]	valid_set's l2: 1223.44	valid_set's rmse: -34.9778
[650]	valid_set's l2: 1196.19	valid_set's rmse: -34.5859
[700]	valid_set's l2: 1174.36	valid_set's rmse: -34.2689
[750]	valid_set's l2: 1158.46	valid_set's rmse: -34.0362
[800]	valid_set's l2: 1135.22	valid_set's rmse: -33.693
[850]	valid_set's l2: 1120.43	valid_set's rmse: -33.4728
[900]	valid_set's l2: 1106.72	vali

Saving C:\Users\Lev\Documents\GitHub\Diploma\models\autogluon_gpu\models\LightGBM\model.pkl
Saving C:\Users\Lev\Documents\GitHub\Diploma\models\autogluon_gpu\utils\attr\LightGBM\y_pred_proba_val.pkl
	-25.0166	 = Validation score   (-rmse)
	411.76s	 = Training   runtime
	1.75s	 = Validation runtime
	27802.9	 = Inference  throughput (rows/s | 48644 batch size)
Saving C:\Users\Lev\Documents\GitHub\Diploma\models\autogluon_gpu\models\trainer.pkl
Fitting model: RandomForestMSE ... Training model for up to 2699.25s of the 2699.22s of remaining time.
	Fitting RandomForestMSE with 'num_gpus': 1, 'num_cpus': 12
	Fitting with cpus=12, gpus=1, mem=3.0/16.5 GB
	Time limit exceeded... Skipping RandomForestMSE.
Saving C:\Users\Lev\Documents\GitHub\Diploma\models\autogluon_gpu\models\trainer.pkl
Fitting model: CatBoost ... Training model for up to 2187.58s of the 2187.55s of remaining time.
	Fitting CatBoost with 'num_gpus': 1, 'num_cpus': 6
	Fitting with cpus=6, gpus=1, mem=14.4/10.6 GB
	Training Ca

0:	learn: 96.2439146	test: 97.7397049	best: 97.7397049 (0)	total: 189ms	remaining: 189ms
1:	learn: 94.6059232	test: 96.1100567	best: 96.1100567 (1)	total: 209ms	remaining: 0us
bestTest = 96.1100567
bestIteration = 1
0:	learn: 96.2439146	test: 97.7397049	best: 97.7397049 (0)	total: 19.7ms	remaining: 8.55s
20:	learn: 76.3077102	test: 77.7948572	best: 77.7948572 (20)	total: 362ms	remaining: 7.12s
40:	learn: 68.2850311	test: 69.7917892	best: 69.7917892 (40)	total: 682ms	remaining: 6.54s
60:	learn: 63.8300021	test: 65.4681609	best: 65.4681609 (60)	total: 1s	remaining: 6.12s
80:	learn: 60.9560005	test: 62.6700138	best: 62.6700138 (80)	total: 1.32s	remaining: 5.74s
100:	learn: 58.7401721	test: 60.5020712	best: 60.5020712 (100)	total: 1.64s	remaining: 5.4s
120:	learn: 57.0125018	test: 58.8490716	best: 58.8490716 (120)	total: 1.94s	remaining: 5.01s
140:	learn: 55.6027335	test: 57.4984882	best: 57.4984882 (140)	total: 2.25s	remaining: 4.68s
160:	learn: 54.4500874	test: 56.3650541	best: 56.365054

Saving C:\Users\Lev\Documents\GitHub\Diploma\models\autogluon_gpu\models\CatBoost\model.pkl
Saving C:\Users\Lev\Documents\GitHub\Diploma\models\autogluon_gpu\utils\attr\CatBoost\y_pred_proba_val.pkl
	-48.3775	 = Validation score   (-rmse)
	32.39s	 = Training   runtime
	0.02s	 = Validation runtime
	2533656.1	 = Inference  throughput (rows/s | 48644 batch size)
Saving C:\Users\Lev\Documents\GitHub\Diploma\models\autogluon_gpu\models\trainer.pkl
Fitting model: ExtraTreesMSE ... Training model for up to 2155.16s of the 2155.13s of remaining time.
	Fitting ExtraTreesMSE with 'num_gpus': 1, 'num_cpus': 12
	Fitting with cpus=12, gpus=1, mem=3.0/10.4 GB
	Time limit exceeded... Skipping ExtraTreesMSE.
Saving C:\Users\Lev\Documents\GitHub\Diploma\models\autogluon_gpu\models\trainer.pkl
Fitting model: NeuralNetFastAI ... Training model for up to 1528.97s of the 1528.93s of remaining time.
	Fitting NeuralNetFastAI with 'num_gpus': 1, 'num_cpus': 6
	To force training the model, specify the model hy

[0]	validation_0-rmse:94.82586	validation_0-_rmse:94.82586
[50]	validation_0-rmse:51.99025	validation_0-_rmse:51.99025
[100]	validation_0-rmse:47.30223	validation_0-_rmse:47.30224
[150]	validation_0-rmse:45.09689	validation_0-_rmse:45.09689
[200]	validation_0-rmse:43.40139	validation_0-_rmse:43.40139
[250]	validation_0-rmse:41.86244	validation_0-_rmse:41.86244
[300]	validation_0-rmse:40.62485	validation_0-_rmse:40.62485
[350]	validation_0-rmse:39.81652	validation_0-_rmse:39.81652
[400]	validation_0-rmse:39.24174	validation_0-_rmse:39.24174
[450]	validation_0-rmse:38.55909	validation_0-_rmse:38.55909
[500]	validation_0-rmse:37.89528	validation_0-_rmse:37.89528
[550]	validation_0-rmse:37.47984	validation_0-_rmse:37.47984
[600]	validation_0-rmse:37.02473	validation_0-_rmse:37.02473
[650]	validation_0-rmse:36.65001	validation_0-_rmse:36.65001
[700]	validation_0-rmse:36.25955	validation_0-_rmse:36.25955
[750]	validation_0-rmse:35.93123	validation_0-_rmse:35.93123
[800]	validation_0-rmse:35.

C:\Users\Lev\Miniconda3\envs\Diploma\lib\site-packages\xgboost\core.py:158: UserWarning: [18:41:18] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)
Saving C:\Users\Lev\Documents\GitHub\Diploma\models\autogluon_gpu\models\XGBoost\model.pkl
Saving C:\Users\Lev\Documents\GitHub\Diploma\models\autogluon_gpu\utils\attr\XGBoost\y_pred_proba_val.pkl
	-25.8509	 = Validation score   (-rmse)
	184.45s	 = Training   runtime
	0.74s	 = Validation runtime
	66143.4	 = Inference  throu

In [9]:
val = pd.read_parquet("../data/money_calc/val.parquet")
test = pd.read_parquet("../data/money_calc/test.parquet")

val = val.reset_index(drop=True)
test = test.reset_index(drop=True)

val, val_meta = optimize_df_for_memory(val)
test, test_meta = optimize_df_for_memory(test)

In [10]:
lb = predictor.leaderboard(val, silent=True)
print(lb)

val_pred = predictor.predict(val.drop(columns=[y_col]))
test_pred = predictor.predict(test.drop(columns=[y_col]))

print("\nValidation metrics")
print("SMAPE:", smape(val[y_col], val_pred))
print("RMSE :", rmse(val[y_col], val_pred))
print("MAPE :", mape(val[y_col], val_pred))

print("\nTest metrics")
print("SMAPE:", smape(test[y_col], test_pred))
print("RMSE :", rmse(test[y_col], test_pred))
print("MAPE :", mape(test[y_col], test_pred))

Loading: C:\Users\Lev\Documents\GitHub\Diploma\models\autogluon_gpu\models\LightGBMXT\model.pkl
Loading: C:\Users\Lev\Documents\GitHub\Diploma\models\autogluon_gpu\models\LightGBM\model.pkl
Loading: C:\Users\Lev\Documents\GitHub\Diploma\models\autogluon_gpu\models\CatBoost\model.pkl
Loading: C:\Users\Lev\Documents\GitHub\Diploma\models\autogluon_gpu\models\XGBoost\model.pkl
Loading: C:\Users\Lev\Documents\GitHub\Diploma\models\autogluon_gpu\models\NeuralNetTorch\model.pkl
Loading: C:\Users\Lev\Documents\GitHub\Diploma\models\autogluon_gpu\models\WeightedEnsemble_L2\model.pkl


                 model  score_test  score_val eval_metric  pred_time_test  \
0             LightGBM  -62.287696 -25.016630        rmse        8.934835   
1  WeightedEnsemble_L2  -63.880265 -24.738087        rmse       28.443712   
2           LightGBMXT  -64.869122 -26.727402        rmse        9.640640   
3              XGBoost  -71.330790 -25.850865        rmse       19.489686   
4             CatBoost  -73.899174 -48.377465        rmse        0.137606   
5       NeuralNetTorch  -74.394532 -39.659062        rmse        1.657629   

   pred_time_val     fit_time  pred_time_test_marginal  \
0       1.749603   411.762312                 8.934835   
1       2.485035   596.267825                 0.019191   
2       1.873203   442.478833                 9.640640   
3       0.735432   184.449996                19.489686   
4       0.019199    32.387054                 0.137606   
5       0.215604  1335.292608                 1.657629   

   pred_time_val_marginal  fit_time_marginal  stack_l

Loading: C:\Users\Lev\Documents\GitHub\Diploma\models\autogluon_gpu\models\LightGBM\model.pkl
Loading: C:\Users\Lev\Documents\GitHub\Diploma\models\autogluon_gpu\models\XGBoost\model.pkl
Loading: C:\Users\Lev\Documents\GitHub\Diploma\models\autogluon_gpu\models\WeightedEnsemble_L2\model.pkl
Loading: C:\Users\Lev\Documents\GitHub\Diploma\models\autogluon_gpu\models\LightGBM\model.pkl
Loading: C:\Users\Lev\Documents\GitHub\Diploma\models\autogluon_gpu\models\XGBoost\model.pkl
Loading: C:\Users\Lev\Documents\GitHub\Diploma\models\autogluon_gpu\models\WeightedEnsemble_L2\model.pkl



Validation metrics
SMAPE: 0.18561247
RMSE : 63.88026481177011
MAPE : 3173306.2

Test metrics
SMAPE: 0.18806851
RMSE : 71.15784885116328
MAPE : 0.3152462


In [9]:
importance = predictor.feature_importance(train)
importance

These features in provided data are not utilized by the predictor and will be ignored: ['EIC-код_62Z1052783389048', 'EIC-код_62Z1426991213062', 'EIC-код_62Z2462903940798', 'EIC-код_62Z2483925041819', 'EIC-код_62Z3459585554968', 'EIC-код_62Z3584090129468', 'EIC-код_62Z4437758189134', 'EIC-код_62Z4948939192262', 'EIC-код_62Z5821574095639', 'EIC-код_62Z5956494916290', 'EIC-код_62Z5987473505748', 'EIC-код_62Z6337618908569', 'EIC-код_62Z6634912359403', 'EIC-код_62Z6814717943705', 'EIC-код_62Z771900708749Y', 'EIC-код_nan', 'АЗС_АЗС_100', 'АЗС_АЗС_69', 'АЗС_АЗС_70', 'АЗС_АЗС_78', 'АЗС_АЗС_83', 'АЗС_АЗС_84', 'АЗС_АЗС_86', 'АЗС_АЗС_87', 'АЗС_АЗС_88', 'АЗС_АЗС_89', 'АЗС_АЗС_90', 'АЗС_АЗС_901', 'АЗС_АЗС_902', 'АЗС_АЗС_91', 'АЗС_АЗС_911', 'АЗС_АЗС_92', 'АЗС_АЗС_93', 'АЗС_АЗС_94', 'АЗС_АЗС_95', 'АЗС_АЗС_99', 'АЗС_nan', 'Тип_ОККО-LPG', 'Тип_nan', 'Область_Чернівецька', 'Область_nan', 'ОСР код_MGA-00200', 'ОСР код_MGA-00400', 'ОСР код_MGA-00500', 'ОСР код_MGA-00600', 'ОСР код_MGA-00700', 'ОСР код_MGA

,importance,stddev,p_value,n,p99_high,p99_low
Hour,0.200087,0.002841,4.872958e-09,5,0.205935,0.194238
GPS-координати - Довгота,0.163726,0.003951,4.066791e-08,5,0.171861,0.155591
apparent_temperature,0.159431,0.009788,1.696316e-06,5,0.179585,0.139277
dew_point_2m,0.112861,0.003281,8.559799e-08,5,0.119617,0.106106
Тип_ОККО-комплекс,0.101933,0.002160,2.419507e-08,5,0.106381,0.097485
...,...,...,...,...,...,...
Область_Одеська,-0.000002,0.000005,7.436924e-01,5,0.000008,-0.000011
АЗС_АЗС_19,-0.000002,0.000018,6.113841e-01,5,0.000035,-0.000040
АЗС_АЗС_68,-0.000019,0.000076,6.959440e-01,5,0.000138,-0.000175
EIC-код_62Z3522594292484,-0.000019,0.000090,6.717813e-01,5,0.000166,-0.000205
